In [6]:
from pathlib import Path
import os
import pandas as pd
# from helpers.utils import dice_score
from helpers import paths
from helpers.paths import (
    RESOURCES_DIR, TRAIN_ROOT, PROJECT_ROOT, DATA_ROOT
)
from reload_recursive import reload_recursive
from helpers.shell_interface import open_itksnap_workspace_cmd

In [ ]:
subject_sessions = pd.read_csv(RESOURCES_DIR/"subject-sessions.csv", index_col="sub")
subjects_file = PROJECT_ROOT / "training/roi_train2/subjects.txt"
with open(subjects_file, 'r') as f:
    subjects = [line.strip() for line in f.readlines()]
    
prl_df = pd.read_csv(
    "/home/srs-9/Projects/prl_project/src/resources/PRL_spreadsheet-lstai_update_label_reference.csv",
    index_col="subid"
)



In [3]:
from collections import defaultdict
dataroot = Path("/media/smbshare/3Tpioneer_bids")
label_names = [
    "lesion.t3m20/prl_mask_def_prob_LR.nii.gz", 
    "lesion.t3m20/prl_mask_def_prob_CH.nii.gz",
    "lesion.t3m20/prl_mask_def_prob_SRS.nii.gz",
    "lesion.t3m20/prl_mask_def_prob_SRS_CH.nii.gz",
]

labels_to_check = defaultdict(list)
for subid in subjects:
    sesid = subject_sessions.loc[int(subid), "ses"]
    subject_root = dataroot / f"sub-ms{subid}" / f"ses-{sesid}"
    for n in label_names:
        p = subject_root / n
        if p.exists():
            labels_to_check[subid].append(p)

In [4]:
for k,v in labels_to_check.copy().items():
    if len(v) < 2:
        labels_to_check.pop(k)

In [5]:
labels_to_check.keys()

dict_keys(['1038', '1074', '1080', '1215', '1293', '1348', '1396', '2026', '2041'])

In [12]:
import subprocess
subid = "1215"
labels = labels_to_check[subid]
subject_root = labels[0].parent.parent
images = [subject_root / im for im in ["flair.nii.gz", "phase.nii.gz"]]
for lab in labels:
    print(lab.name)
cmd = open_itksnap_workspace_cmd(images, labels, rename_root=("/mnt/z", "Z:/"))
# subprocess.run(cmd, shell=True)
print(cmd)

prl_mask_def_prob_CH.nii.gz
prl_mask_def_prob_SRS.nii.gz
itksnap -g Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/flair.nii.gz -o Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/phase.nii.gz -s Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/lesion.t3m20/prl_mask_def_prob_CH.nii.gz Z:/3Tpioneer_bids/sub-ms1215/ses-20180429/lesion.t3m20/prl_mask_def_prob_SRS.nii.gz


- 2026: SRS and CH (LR not subsegmented); 2 PRL [2, 1]
- 1038: CH and LR; 2 PRL but part of same confluent lesion complex [1]
- 1080: SRS and LR; 2 PRL [3, 1]
- 1215: SRS and CH; 2 PRL; [2, 3]
- 1348: LR and SRS; 2 PRL [1, 6]
- 1396: LR, SRS, and CH; 1 PRL [1]
- 1011: LR is not subsegmented
- 1074: SRS is not subsegmented
- 1293: Doesn't have unique labels (LR, LR_SRS, LR_SRS_CH)
- 2041: CH is not subsegmented

In [8]:
import preprocessing.create_rois
reload_recursive(preprocessing.create_rois)
from preprocessing.create_rois import ensure_ring_seg

work_folder = Path("/home/srs-9/Projects/prl_project/notebooks/interrater_reliability")
work_folder.mkdir(exist_ok=True)

In [9]:
subid = 2026
prl_initials = ["CH", "SRS"]
ses = str(subject_sessions.loc[subid, 'ses'])
subject_dir = DATA_ROOT / f"sub{subid}-{ses}"

# PRL1
lesion_ind = int(prl_df.loc[subid, "PRL1_label"])
print(lesion_ind)
lesion_folder = subject_dir / str(lesion_ind)
ensure_ring_seg(subject_dir, suffix="CH", output_dir=work_folder)

2


CalledProcessError: Command 'fslmaths /media/smbshare/srs-9/prl_project/data/sub2026-20160917/prl_mask_def_prob_CH.nii.gz -thr 2 -uthr 2 /home/srs-9/Projects/prl_project/notebooks/interrater_reliability/prl_rim_def_prob_CH.nii.gz' returned non-zero exit status 127.

In [14]:
print(lesion_folder)

/media/smbshare/srs-9/prl_project/data/sub2026-20160917/2
